<a href="https://colab.research.google.com/github/SmallChungus1/computer_vision_awesome_notebooks/blob/main/spanish_stable_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install multilingual-clip torch
!pip install transformers==4.49
!pip install diffusers==0.32.2
#downgrade transformers and diffusers to avoid clip text error https://github.com/huggingface/diffusers/issues/12436

  Using cached diffusers-0.32.2-py3-none-any.whl.metadata (18 kB)
Using cached diffusers-0.32.2-py3-none-any.whl (3.2 MB)
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.35.1
    Uninstalling diffusers-0.35.1:
      Successfully uninstalled diffusers-0.35.1


In [12]:
from transformers import ClapProcessor, ClapAudioModel
from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image
import torch
import time
import librosa
import torch.nn.functional as F
import torch.nn as nn
from PIL import Image
import numpy as np
from multilingual_clip import pt_multilingual_clip
import transformers
import types

device = device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using devices: {device}")

using devices: cuda


In [18]:
#monkey patch function to get embeddings before linear pooling layer
#source code: https://github.com/FreddeFrallan/Multilingual-CLIP/blob/main/multilingual_clip/pt_multilingual_clip.py
def get_last_hidden_state(self, txt, tokenizer):
    txt_tok = tokenizer(txt, padding=True, return_tensors='pt')
    outputs = self.transformer(**txt_tok)[0]
    return outputs


In [19]:
multi_lingual_clip_name = 'M-CLIP/XLM-Roberta-Large-Vit-L-14'

# Load Model & Tokenizer
model = pt_multilingual_clip.MultilingualCLIP.from_pretrained(multi_lingual_clip_name)
#apply patch
model.get_last_hidden_state = types.MethodType(get_last_hidden_state, model)

tokenizer = transformers.AutoTokenizer.from_pretrained(multi_lingual_clip_name)

In [37]:
#sd_model_name = "stable-diffusion-v1-5/stable-diffusion-v1-5" #768 token embed dim
sd_model_name = "stabilityai/sd-turbo" #1024 token embed dim
sd_model = StableDiffusionPipeline.from_pretrained(sd_model_name, torch_dtype=torch.float16).to(device)

model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.46G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [38]:
#https://en.wikipedia.org/wiki/List_of_Mexican_dishes
texts = [
   'tacos de pescado',
   'Tortilla española',
   'Gambas al ajillo',
   "tacos al pastor"]
embeddings = model.forward(texts, tokenizer)
last_layer_embeds = model.get_last_hidden_state(texts, tokenizer)
print(embeddings.shape)
print(last_layer_embeds.shape)

torch.Size([4, 768])
torch.Size([4, 7, 1024])


In [40]:
img_cap_pairs = {}

for text, embeds in zip(texts, last_layer_embeds):
    print(text)
    if embeds.ndim == 2:
        embeds = embeds.unsqueeze(0)
    # print(embeds.shape)
    gen_img = sd_model(prompt_embeds=embeds)
    img_cap_pairs[text] = gen_img[0][0]

tacos de pescado


  0%|          | 0/50 [00:00<?, ?it/s]

Tortilla española


  0%|          | 0/50 [00:00<?, ?it/s]

Gambas al ajillo


  0%|          | 0/50 [00:00<?, ?it/s]

tacos al pastor


  0%|          | 0/50 [00:00<?, ?it/s]

In [41]:
for text, img in img_cap_pairs.items():
    img.save(f"{text}.png")